# Tiền xử lý dữ liệu (Lần đầu)
#### Tập dữ liệu: NIH Chest X-Ray 14

## 1.  Thiết lập
### 1.1 Khai báo thư viện + Biến cố định

In [24]:
# =========================
# Khai báo thư viện
# =========================

from pathlib import Path
import json

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

# =========================
# Đường dẫn cố định
# =========================

ROOT_DIRECTORY = (Path.cwd().parent).resolve()

### 1.2. Cài đặt chung

In [25]:
IMAGE_DIRECTORY             = (ROOT_DIRECTORY / "original_dataset" / "NIH ChestX-Ray14" / "image").resolve()
TRAIN_CSV_DIRECTORY         = (ROOT_DIRECTORY / "original_dataset" / "NIH ChestX-Ray14" / "train.csv").resolve()
VALIDATION_CSV_DIRECTORY    = (ROOT_DIRECTORY / "original_dataset" / "NIH ChestX-Ray14" / "validation.csv").resolve()
TEST_CSV_DIRECTORY          = (ROOT_DIRECTORY / "original_dataset" / "NIH ChestX-Ray14" / "test.csv").resolve()
OUTPUT_DIRECTORY            = (ROOT_DIRECTORY / "dataset").resolve()

RESIZE_IMAGES               = True
IMAGE_SIZE                  = 224

ID_COLUMN                   = "Image Index"
LABEL_COLUMN                = "Finding Labels"
PATIENT_ID_COLUMN           = "Patient ID"
NO_FINDING_LABEL            = "No Finding"
LABEL_SEPARATOR             = "|"
    

### 1.3. Kiểm tra thông tin cài đặt

In [26]:
MAX_TITLE_WIDTH = 25
MAX_MESSAGE_WIDTH = 11

paths_to_check = {
    "ROOT_DIRECTORY"           : ROOT_DIRECTORY,
    "IMAGE_DIRECTORY"          : IMAGE_DIRECTORY,
    "TRAIN_CSV_DIRECTORY"      : TRAIN_CSV_DIRECTORY,
    "VALIDATION_CSV_DIRECTORY" : VALIDATION_CSV_DIRECTORY,
    "TEST_CSV_DIRECTORY"       : TEST_CSV_DIRECTORY,
}

print("=" * 80)
print("KIỂM TRA CẤU HÌNH TIỀN XỬ LÝ DỮ LIỆU")
print("=" * 80)

all_valid = True

# =========================
# Kiểm tra đường dẫn dũ liệu đầu vào
# =========================

print("\n[1] KIỂM TRA ĐƯỜNG DẪN DỮ LIỆU ĐẦU VÀO")
print("-" * 80)

for name, path in paths_to_check.items():
    exists = path.exists()

    if exists:
        if path.is_dir():
            
            path_type = "Directory"
            
        elif path.is_file():
            
            path_type = "File"
            
        else:
            
            path_type = "Unknown"

        print(f"{'[OK]'     :<{MAX_MESSAGE_WIDTH}} {name:<{MAX_TITLE_WIDTH}}: Type: {path_type:<10} : {path}")
        
    else:
        
        print(f"{'[MISSING]':<{MAX_MESSAGE_WIDTH}} {name:<{MAX_TITLE_WIDTH}}: {path}")
        all_valid = False

# =========================
# Kiểm tra cấu hình cột và nhãn
# =========================

print("\n[2] KIỂM TRA CẤU HÌNH CỘT VÀ NHÃN")
print("-" * 80)

    ## ---------------------------------------------------------
    ## Kiểm tra cột dữ liệu
    ## ---------------------------------------------------------

# =========================
# Kiểm tra cấu hình cột và nhãn
# =========================

column_configs = {
    "ID_COLUMN"         : ID_COLUMN,
    "LABEL_COLUMN"      : LABEL_COLUMN,
    "PATIENT_ID_COLUMN" : PATIENT_ID_COLUMN,
    "LABEL_SEPARATOR"   : LABEL_SEPARATOR,
    "NO_FINDING_LABEL"  : NO_FINDING_LABEL,
}

for name, value in column_configs.items():

    if isinstance(value, str) and value.strip():

        print(
            f"{'[OK]':<{MAX_MESSAGE_WIDTH}} "
            f"{name:<{MAX_TITLE_WIDTH}}: {value}"
        )

    else:

        print(
            f"{'[INVALID]':<{MAX_MESSAGE_WIDTH}} "
            f"{name:<{MAX_TITLE_WIDTH}}: "
            f"Phải là chuỗi không rỗng."
        )

        all_valid = False


# =========================
# Kiểm tra tên cột trùng nhau
# =========================

required_column_names = [
    ID_COLUMN,
    LABEL_COLUMN,
    PATIENT_ID_COLUMN,
]

if len(required_column_names) == len(set(required_column_names)):

    print(
        f"{'[OK]':<{MAX_MESSAGE_WIDTH}} "
        f"{'COLUMN_NAMES':<{MAX_TITLE_WIDTH}}: "
        f"Không trùng nhau"
    )

else:

    print(
        f"{'[INVALID]':<{MAX_MESSAGE_WIDTH}} "
        f"{'COLUMN_NAMES':<{MAX_TITLE_WIDTH}}: "
        f"Có tên cột bị trùng"
    )

    all_valid = False


# =========================
# Kiểm tra tệp CSV (Dữ liệu thường)
# =========================

print("\n[3] KIỂM TRA DỮ LIỆU TRONG TỆP TIN CSV")
print("-" * 80)

csv_files = {
    "TRAIN"      : TRAIN_CSV_DIRECTORY,
    "VALIDATION" : VALIDATION_CSV_DIRECTORY,
    "TEST"       : TEST_CSV_DIRECTORY,
}

required_columns = {ID_COLUMN, LABEL_COLUMN, PATIENT_ID_COLUMN,}

loaded_dataframes = {}


for split_name, csv_path in csv_files.items():

    try:

        df = pd.read_csv(csv_path)
        loaded_dataframes[split_name] = df
        missing_columns = (required_columns - set(df.columns))

        if missing_columns:

            print(f"{'[INVALID]':<{MAX_MESSAGE_WIDTH}} {split_name:<{MAX_TITLE_WIDTH}}: Thiếu cột {sorted(missing_columns)}")
            all_valid = False
            continue
        
        # =========================
        # Kiểm tra dữ liệu
        # =========================

        checks = {
            "IMAGE ID THIẾU":
                df[ID_COLUMN].isna().sum(),

            "IMAGE ID TRÙNG":
                df[ID_COLUMN].duplicated().sum(),

            "PATIENT ID THIẾU":
                df[PATIENT_ID_COLUMN].isna().sum(),

            "LABEL THIẾU":
                df[LABEL_COLUMN].isna().sum(),

            "LABEL RỖNG":
                (df[LABEL_COLUMN].fillna("").astype(str).str.strip().eq("").sum()),
        }
        
        invalid_count = sum(checks.values())

        if invalid_count == 0:

            print(f"{'[OK]':<{MAX_MESSAGE_WIDTH}} {split_name:<{MAX_TITLE_WIDTH}}: {len(df):,} dòng - dữ liệu hợp lệ")

        else:

            print(f"{'[INVALID]':<{MAX_MESSAGE_WIDTH}} {split_name:<{MAX_TITLE_WIDTH}}: Dữ liệu không hợp lệ")

            for check_name, count in checks.items():

                if count > 0:

                    print(f"{'':<{MAX_MESSAGE_WIDTH}} {check_name:<{MAX_TITLE_WIDTH}}: {count:,}")

            all_valid = False


    except Exception as error:

        print(f"{'[INVALID]':<{MAX_MESSAGE_WIDTH}} {split_name:<{MAX_TITLE_WIDTH}}: Không đọc được CSV: {error}")
        all_valid = False


# =========================
# Kiểm tra No Finding
# =========================

print("\n[4] KIỂM TRA NO FINDING")
print("-" * 80)

no_finding_total = 0
invalid_no_finding_total = 0


for split_name, df in loaded_dataframes.items():

    if LABEL_COLUMN not in df.columns:
        continue

    labels_series = (df[LABEL_COLUMN].fillna("").astype(str).str.strip())
    no_finding_count = (labels_series.eq(NO_FINDING_LABEL).sum())

    invalid_count = (
        labels_series
        .str.split(LABEL_SEPARATOR, regex=False)
        .apply(
            lambda labels: (NO_FINDING_LABEL in labels and len(labels) > 1)
        )
        .sum()
    )

    no_finding_total += no_finding_count
    invalid_no_finding_total += invalid_count

    status = (
        "[OK]"
        if invalid_count == 0
        else "[INVALID]"
    )

    print(f"{status:<{MAX_MESSAGE_WIDTH}} {split_name:<{MAX_TITLE_WIDTH}}: No Finding = {no_finding_count:,}")

    if invalid_count > 0:

        print(f"{'':<{MAX_MESSAGE_WIDTH}} {'FORMAT INVALID':<{MAX_TITLE_WIDTH}}: {invalid_count:,} dòng")
        all_valid = False

print(f"{'[INFO]':<{MAX_MESSAGE_WIDTH}} {'TOTAL NO FINDING':<{MAX_TITLE_WIDTH}}: {no_finding_total:,}")

# =========================
# Dò class từ tập dữ liệu
# =========================

print("\n[5] CÁC LỚP BỆNH TRONG TẬP DỮ LIỆU")
print("-" * 80)

CLASSES = []

if (
    "TRAIN" in loaded_dataframes
    and LABEL_COLUMN in loaded_dataframes["TRAIN"].columns
):

    CLASSES = sorted({
        label.strip()
        for value in (loaded_dataframes["TRAIN"][LABEL_COLUMN].dropna().astype(str))
        for label in value.split(LABEL_SEPARATOR)
        if (label.strip() and label.strip() != NO_FINDING_LABEL)
    })


    if CLASSES:

        print(f"{'[OK]':<{MAX_MESSAGE_WIDTH}} {'Phát hiện lớp':<{MAX_TITLE_WIDTH}}: {len(CLASSES)} lớp")
        print('\n')
        
        for index, label in enumerate(CLASSES, start=1):

            print(f"{'':<{MAX_MESSAGE_WIDTH}} {index:>2}. {label}")

    else:

        print(f"{'[INVALID]':<{MAX_MESSAGE_WIDTH}} {'Phát hiện lớp':<{MAX_TITLE_WIDTH}}: Không tìm thấy lớp bệnh")
        all_valid = False

else:

    print(f"{'[INVALID]':<{MAX_MESSAGE_WIDTH}} {'Phát hiện lớp':<{MAX_TITLE_WIDTH}}: Không đọc được TRAIN")
    all_valid = False
        
# =========================
# Kiểm tra thư mục đầu ra
# =========================

print("\n[6] KIỂM TRA THƯ MỤC ĐẦU RA")
print("-" * 80)

if not OUTPUT_DIRECTORY.exists():
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    
    print(f"{'[CREATED]':<{MAX_MESSAGE_WIDTH}} {'OUTPUT_DIRECTORY': <25}: {OUTPUT_DIRECTORY}")
    
else:
    
    print(f"{'[OK]'     :<{MAX_MESSAGE_WIDTH}} {'OUTPUT_DIRECTORY': <25}: {OUTPUT_DIRECTORY}")

# =========================
# Kiểm tra dữ liệu
# =========================

print("\n[7] KIỂM TRA DỮ LIỆU")
print("-" * 80)

if IMAGE_DIRECTORY.exists() and IMAGE_DIRECTORY.is_dir():

    image_extensions = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff",}

    image_files = [
        file
        for file in IMAGE_DIRECTORY.rglob("*")
        if file.is_file()
        and file.suffix.lower() in image_extensions
    ]

    num_images = len(image_files)

    print(f"{'[INFO]':<{MAX_MESSAGE_WIDTH}} {'Số lượng ảnh tìm thấy':<{MAX_TITLE_WIDTH}}: {num_images:,}")

    if num_images == 0:
        
        print(f"{'[INVALID]':<{MAX_MESSAGE_WIDTH}} Không tìm thấy ảnh trong đường dẫn IMAGE_DIRECTORY.")
        all_valid = False
        
    else:
        
        print(f"{'[OK]'     :<{MAX_MESSAGE_WIDTH}} Dữ liệu ảnh tồn tại.")

else:
    
    print(f"{'[INVALID]'    :<{MAX_MESSAGE_WIDTH}} IMAGE_DIRECTORY không tồn tại hoặc không phải thư mục.")
    all_valid = False

# =========================
# Những thiết lập khác
# =========================

print("\n[8] KIỂM TRA CẤU HÌNH KHÁC")
print("-" * 80)

# RESIZE_IMAGES
if isinstance(RESIZE_IMAGES, bool):
    
    print(f"{'[OK]'     :<{MAX_MESSAGE_WIDTH}} {'RESIZE_IMAGES':<{MAX_TITLE_WIDTH}}: {RESIZE_IMAGES}")
    
else:
    
    print(f"{'[INVALID]':<{MAX_MESSAGE_WIDTH}} RESIZE_IMAGES phải là bool (True/False), nhưng nhận được: {RESIZE_IMAGES}")
    all_valid = False

# IMAGE_SIZE
if isinstance(IMAGE_SIZE, int) and not isinstance(IMAGE_SIZE, bool):

    if IMAGE_SIZE > 0:
        
        print(f"{'[OK]'     :<{MAX_MESSAGE_WIDTH}} {'IMAGE_SIZE':<{MAX_TITLE_WIDTH}}: {IMAGE_SIZE} x {IMAGE_SIZE}")
        
    else:
        
        print(f"{'[INVALID]':<{MAX_MESSAGE_WIDTH}} IMAGE_SIZE phải lớn hơn 0, nhưng nhận được: {IMAGE_SIZE}")
        all_valid = False

else:
    
    print(f"{'[INVALID]'    :<{MAX_MESSAGE_WIDTH}} IMAGE_SIZE phải là số nguyên, nhưng nhận được: {IMAGE_SIZE}")
    all_valid = False

# Hiển thị cấu hình resize thực tế
if (
    isinstance(RESIZE_IMAGES, bool)
    and RESIZE_IMAGES
    and isinstance(IMAGE_SIZE, int)
    and not isinstance(IMAGE_SIZE, bool)
    and IMAGE_SIZE > 0
):
    RESIZED_DIRECTORY = OUTPUT_DIRECTORY / f"images_{IMAGE_SIZE}"

    if not RESIZED_DIRECTORY.exists():
        RESIZED_DIRECTORY.mkdir(parents=True, exist_ok=True)
        
        print(f"{'[CREATED]':<{MAX_MESSAGE_WIDTH}} {'RESIZED_DIRECTORY':<{MAX_TITLE_WIDTH}}: {RESIZED_DIRECTORY}")
        
    else:
        
        print(f"{'[OK]'     :<{MAX_MESSAGE_WIDTH}} {'RESIZED_DIRECTORY':<{MAX_TITLE_WIDTH}}: {RESIZED_DIRECTORY}")

elif isinstance(RESIZE_IMAGES, bool) and not RESIZE_IMAGES:
    
    print(f"{'[INFO]'      :<{MAX_MESSAGE_WIDTH}} Resize ảnh đang tắt. Ảnh sẽ giữ kích thước gốc.")

# =========================
# Thông báo
# =========================

print("\n" + "=" * 80)

if all_valid:
    
    print("TẤT CẢ DỮ LIỆU VÀ CẤU HÌNH ĐỀU HỢP LỆ.")
    
else:
    
    print("PHÁT HIỆN DỮ LIỆU HOẶC CẤU HÌNH KHÔNG HỢP LỆ.")
    print("KIỂM TRA CÁC MỤC [MISSING] / [INVALID] Ở TRÊN.")

print("=" * 80)

assert all_valid, "Thiết lập không hợp lệ."

KIỂM TRA CẤU HÌNH TIỀN XỬ LÝ DỮ LIỆU

[1] KIỂM TRA ĐƯỜNG DẪN DỮ LIỆU ĐẦU VÀO
--------------------------------------------------------------------------------
[OK]        ROOT_DIRECTORY           : Type: Directory  : E:\chest_classification
[OK]        IMAGE_DIRECTORY          : Type: Directory  : E:\chest_classification\original_dataset\NIH ChestX-Ray14\image
[OK]        TRAIN_CSV_DIRECTORY      : Type: File       : E:\chest_classification\original_dataset\NIH ChestX-Ray14\train.csv
[OK]        VALIDATION_CSV_DIRECTORY : Type: File       : E:\chest_classification\original_dataset\NIH ChestX-Ray14\validation.csv
[OK]        TEST_CSV_DIRECTORY       : Type: File       : E:\chest_classification\original_dataset\NIH ChestX-Ray14\test.csv

[2] KIỂM TRA CẤU HÌNH CỘT VÀ NHÃN
--------------------------------------------------------------------------------
[OK]        ID_COLUMN                : Image Index
[OK]        LABEL_COLUMN             : Finding Labels
[OK]        PATIENT_ID_COLUMN      

## 2. Mã hóa nhãn dữ liệu
#### Bằng phương pháp One-hot encoding
##### No Finding được đánh dấu bằng cách toàn bộ giá trị trong vector về 0

### 2.1. Hàm mã hóa dữ liệu

In [27]:
def encode_labels(df, classes):

    df = df.copy()

    parsed_labels = (
        df[LABEL_COLUMN]
        .astype(str)
        .str.split(LABEL_SEPARATOR)
    )

    for label in classes:

        df[label] = (
            parsed_labels
            .apply(
                lambda labels:
                int(label in labels)
            )
            .astype(np.uint8)
        )

    # Số bệnh có trong mỗi ảnh
    df["num_positive_labels"] = (
        df[classes]
        .sum(axis=1)
        .astype(np.uint8)
    )

    # No Finding chỉ dùng để đánh dấu,
    # không phải output thứ 15
    df["is_no_finding"] = (
        df[LABEL_COLUMN]
        .eq(NO_FINDING_LABEL)
        .astype(np.uint8)
    )

    return df

### 2.2. Tiến hành mã hóa

In [28]:
train_df = encode_labels(
    loaded_dataframes["TRAIN"],
    CLASSES
)

validation_df = encode_labels(
    loaded_dataframes["VALIDATION"],
    CLASSES
)

test_df = encode_labels(
    loaded_dataframes["TEST"],
    CLASSES
)

print(
    train_df[
        [ID_COLUMN, LABEL_COLUMN]
        + CLASSES
    ].head()
)


        Image Index       Finding Labels  Atelectasis  Cardiomegaly  \
0  00000003_001.png               Hernia            0             0   
1  00000003_002.png               Hernia            0             0   
2  00000003_003.png  Hernia|Infiltration            0             0   
3  00000003_004.png               Hernia            0             0   
4  00000003_005.png               Hernia            0             0   

   Consolidation  Edema  Effusion  Emphysema  Fibrosis  Hernia  Infiltration  \
0              0      0         0          0         0       1             0   
1              0      0         0          0         0       1             0   
2              0      0         0          0         0       1             1   
3              0      0         0          0         0       1             0   
4              0      0         0          0         0       1             0   

   Mass  Nodule  Pleural_Thickening  Pneumonia  Pneumothorax  
0     0       0              

## 3. Ghép tên ảnh với đường dẫn thực tế

### 3.1. Hàm xử lý ghép tên

In [29]:
image_extensions = {
    ".png",
    ".jpg",
    ".jpeg",
    ".bmp",
    ".tif",
    ".tiff",
}

image_map = {
    path.name: path
    for path in IMAGE_DIRECTORY.rglob("*")
    if (
        path.is_file()
        and path.suffix.lower() in image_extensions
    )
}

def attach_image_path(df):

    df = df.copy()

    df["original_path"] = (
        df[ID_COLUMN]
        .map(
            lambda name:
            str(image_map[name])
            if name in image_map
            else None
        )
    )

    return df

### 3.2. Tiến hành xử lý

In [30]:
train_df = attach_image_path(train_df)
validation_df = attach_image_path(validation_df)
test_df = attach_image_path(test_df)

for name, df in {
    "TRAIN": train_df,
    "VALIDATION": validation_df,
    "TEST": test_df,
}.items():

    missing = (
        df["original_path"]
        .isna()
        .sum()
    )

    print(
        f"{name:<12}: "
        f"Missing image = {missing:,}"
    )

    assert missing == 0

TRAIN       : Missing image = 0
VALIDATION  : Missing image = 0
TEST        : Missing image = 0


## 4. Chuyển đổi kích cỡ ảnh
#### Từ 1028 x 1028 --> RESIZE đề ra

## 4.1. Hàm xử lý chuyển đổi kích cỡ ảnh

In [31]:
all_df = pd.concat(
    [
        train_df,
        validation_df,
        test_df,
    ],
    ignore_index=True
)

unique_images = (
    all_df[
        [ID_COLUMN, "original_path"]
    ]
    .drop_duplicates(
        subset=[ID_COLUMN]
    )
    .reset_index(drop=True)
)

resized_paths = {}

for _, row in tqdm(
    unique_images.iterrows(),
    total=len(unique_images),
    desc="Resize"
):

    image_name = row[ID_COLUMN]

    source_path = Path(
        row["original_path"]
    )

    target_path = (
        RESIZED_DIRECTORY
        / image_name
    )

    # Không xử lý lại nếu đã có
    if not target_path.exists():

        with Image.open(source_path) as image:

            image = image.convert("L")

            image = image.resize(
                (
                    IMAGE_SIZE,
                    IMAGE_SIZE
                ),
                Image.Resampling.LANCZOS
            )

            image.save(
                target_path,
                format="PNG"
            )

    resized_paths[
        image_name
    ] = str(
        target_path.resolve()
    )

Resize: 100%|██████████| 112120/112120 [32:32<00:00, 57.43it/s]


In [33]:
for df in [
    train_df,
    validation_df,
    test_df,
]:

    df["image_path"] = (
        df[ID_COLUMN]
        .apply(lambda image_id: str(RESIZED_DIRECTORY / image_id))
    )

## 5. Thống kê dữ liệu

In [34]:
class_statistics = []

for label in CLASSES:

    positive = int(
        train_df[label].sum()
    )

    negative = (
        len(train_df)
        - positive
    )

    positive_rate = (
        positive
        / len(train_df)
    )

    pos_weight = (
        negative / positive
        if positive > 0
        else 1.0
    )

    class_statistics.append({
        "class": label,
        "positive": positive,
        "negative": negative,
        "positive_rate": positive_rate,
        "pos_weight": pos_weight,
    })


class_statistics_df = pd.DataFrame(
    class_statistics
)

display(class_statistics_df)

,class,positive,negative,positive_rate,pos_weight
0,Atelectasis,8193,70741,0.103796,8.634322
1,Cardiomegaly,1940,76994,0.024577,39.687629
2,Consolidation,3262,75672,0.041326,23.198038
3,Edema,1633,77301,0.020688,47.336803
4,Effusion,9227,69707,0.116895,7.554676
5,Emphysema,1922,77012,0.024349,40.068678
6,Fibrosis,1191,77743,0.015089,65.275399
7,Hernia,172,78762,0.002179,457.918605
8,Infiltration,14030,64904,0.177743,4.626087
9,Mass,4101,74833,0.051955,18.247501


## 6. Lưu dữ liệu

In [36]:
train_df.to_csv(
    OUTPUT_DIRECTORY / "train_processed.csv",
    index=False
)

validation_df.to_csv(
    OUTPUT_DIRECTORY / "validation_processed.csv",
    index=False
)

test_df.to_csv(
    OUTPUT_DIRECTORY / "test_processed.csv",
    index=False
)

class_statistics_df.to_csv(
    OUTPUT_DIRECTORY / "class_statistics.csv",
    index=False
)

pos_weight = (
    class_statistics_df[
        "pos_weight"
    ]
    .to_numpy(
        dtype=np.float32
    )
)

np.save(
    OUTPUT_DIRECTORY / "pos_weight.npy",
    pos_weight
)

with open(
    OUTPUT_DIRECTORY / "classes.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        {
            "num_classes": NUM_CLASSES,
            "classes": CLASSES,
            "no_finding": NO_FINDING_LABEL,
            "no_finding_is_class": False,
        },
        file,
        indent=4,
        ensure_ascii=False
    )

NameError: name 'NUM_CLASSES' is not defined